In [1]:
import pandas as pd
from msa.utils.paths import get_joined_dataset
from msa.utils.vars import WINDOW_START, WINDOW_END

df = pd.read_csv(get_joined_dataset(), parse_dates=["article_date"])
observed = set(df["article_date"].dt.normalize().unique())

# Full range of calendar days
full_range = pd.date_range(start=WINDOW_START, end=WINDOW_END, freq="D")

# Missing dates
missing = [d for d in full_range if d.normalize() not in observed]

# Group into contiguous gaps
def get_gaps(dates):
    if not dates:
        return []
    dates = sorted(dates)
    gaps = []
    start = dates[0]
    prev = dates[0]
    for d in dates[1:]:
        if (d - prev).days > 1:
            gaps.append((start, prev))
            start = d
        prev = d
    gaps.append((start, prev))
    return gaps

gap_dates = [d for d in missing if d in full_range]
gaps = get_gaps(gap_dates)

print(f"Desired range: {WINDOW_START} to {WINDOW_END}")
print(f"Total days in range: {len(full_range)}")
print(f"Days with data: {len(observed)}")
print(f"Missing days: {len(missing)}")
print(f"\nGap ranges (start, end):")
for s, e in gaps[:20]:  # first 20 gaps
    print(f"  {s.date()} to {e.date()} ({(e-s).days + 1} days)")
if len(gaps) > 20:
    print(f"  ... and {len(gaps)-20} more gaps")

Desired range: 2024-02-23 to 2026-02-23
Total days in range: 732
Days with data: 107
Missing days: 628

Gap ranges (start, end):
  2024-02-23 to 2024-04-04 (42 days)
  2024-04-10 to 2024-07-01 (83 days)
  2024-07-07 to 2024-07-07 (1 days)
  2024-07-11 to 2024-09-04 (56 days)
  2024-09-10 to 2024-12-04 (86 days)
  2024-12-10 to 2025-03-04 (85 days)
  2025-03-11 to 2025-06-04 (86 days)
  2025-06-10 to 2025-07-04 (25 days)
  2025-07-09 to 2025-09-04 (58 days)
  2025-09-07 to 2025-10-01 (25 days)
  2025-10-04 to 2025-10-05 (2 days)
  2025-10-08 to 2025-12-03 (57 days)
  2025-12-09 to 2025-12-12 (4 days)
  2025-12-16 to 2025-12-18 (3 days)
  2025-12-21 to 2026-01-03 (14 days)
  2026-01-10 to 2026-01-10 (1 days)
